In [ ]:
def create_move_base_option(
    name: str,
    robot: MobileSingleArmPyBulletRobot,
    types: Sequence[Type],
    params_space: Box,
    get_current_base_and_arm_pose: Callable[[SingleArmPyBulletRobot, State, Sequence[Object], Array],
                                            Tuple[Pose, JointPositions]],
    home_orn: Sequence[float],
    collision_bodies: Collection[int],
    seed: int,
    physics_client_id: int,
    held_object_id_at_start: Optional[int] = None,
    ee_to_held_object_transform_at_start: Optional[Tuple[NDArray, NDArray]] = None,
    rng: Optional[np.random.Generator] = None,
    try_arm_only_first: bool = False,
    base_path_planner_max_tries: int = 30,
    workspace_bounds: Optional[Tuple[float, float, float, float]] = None,
    final_finger_state: Optional[float] = None,
    move_to_pose_tol: float = 0.05,     # 5 cm
    vel: float = 0.2,                   # nominal cruise speed
    LOOKAHEAD: float = 0.25,
    wheel_radius: float = 0.065,
    track_width: float = 0.3748,
    force: float = 5.0,                 # (unused here; wheel control path would use it)
    omega_max: float = 17.4,            # wheel joint limit (rad/s)
    orientation_tol: float = 0.06,     # ~5°
    orientation_gain: float = 2.0,      # yaw P gain
    dt: float = 0.045  #10 times the duration of one simulation step in pybullet.
) -> ParameterizedOption:

    # Phase thresholds / gating
    # ORIENTATION_ONLY_DISTANCE = 0.10    # within 10 cm: rotate in place
    # e.g., 0.06–0.07 m
    ORIENTATION_ONLY_DISTANCE = max(0.06, move_to_pose_tol + 0.01)
    ORIENTATION_DEADBAND      = 0.02    # ~1.15° deadband for micro-oscillation
    COMPLETE_STOP_DISTANCE    = move_to_pose_tol

    # Convert wheel limits -> base limits (use these for clipping v, ω)
    v_max = wheel_radius * omega_max
    w_max = 2.0 * wheel_radius * omega_max / track_width


    def _plan_and_cache_base_motion(robot: MobileSingleArmPyBulletRobot, state: State, objects: Sequence[Object],
                                   memory: Dict) -> None:
        
        filtered_collision_bodies = list(collision_bodies)
        if held_object_id_at_start is not None:
            # Exclude the held object from obstacle set
            filtered_collision_bodies = [b for b in filtered_collision_bodies if b != held_object_id_at_start]

        _, table = objects

        table_x, table_y, table_z = (state.get(table, "pose_x"),
                                     state.get(table, "pose_y"),
                                     state.get(table, "pose_z"))

        target_z = table_z * 2.25
        x_workspace = (table_x-0.125, table_x+0.125)
        y_workspace = (table_y-0.2, table_y+0.2)

        x_sample = np.random.uniform(*x_workspace, size=None)
        y_sample = np.random.uniform(*y_workspace, size=None)

        target_ee_pose = Pose(position=(x_sample, y_sample, target_z), orientation=home_orn)
            

        base_path_waypoints: List[Tuple[float, float, float]] = run_coordinated_motion_planning(
                                                                        robot=robot,
                                                                        target_ee_pose=target_ee_pose,
                                                                        collision_bodies=filtered_collision_bodies,
                                                                        seed=seed,
                                                                        physics_client_id=physics_client_id,
                                                                        try_arm_only_first=try_arm_only_first,
                                                                        base_path_planner_max_tries=base_path_planner_max_tries,
                                                                        workspace_bounds=workspace_bounds,
                                                                        rng=np.random.default_rng(seed),
                                                                        final_finger_state=final_finger_state,
                                                                        held_object_id_at_start=held_object_id_at_start,
                                                                        ee_to_held_object_transform_at_start=ee_to_held_object_transform_at_start,
                                                                    )

        if base_path_waypoints is None or len(base_path_waypoints) == 0:
            raise utils.OptionExecutionFailure(f"{name}: Base path planning failed or returned empty path.")

        target_base_pose = base_path_waypoints[-1]
        memory["path"] = base_path_waypoints
        memory["target_base_pose"] = target_base_pose
        memory["path_pointer"] = 0

        return

        

    def _initiable(state: State, memory: dict, objs: Sequence[Object], params: Array) -> bool:
        memory["control_phase"] = "NAVIGATION"
        memory["step_count"] = 0
        return True

    def _policy(state: State, memory: Dict, objects: Sequence[Object], params: Array) -> Action:
        if "path" not in memory or "target_base_pose" not in memory or "path_pointer" not in memory:
            _plan_and_cache_base_motion(robot, state, objects, memory)
        
        
        memory["step_count"] += 1

        # Pose
        # ipdb.set_trace()
        if memory["step_count"] == 1:
            (x, y, theta), arm_q = get_current_base_and_arm_pose(robot, state, objects, params)
            memory["current_internal_base_pose"] = (x,y,theta)
            memory["arm_joints"] = arm_q
        goal_xy = np.array(memory["target_base_pose"][:2])
        goal_yaw = float(memory["target_base_pose"][2])

        cur_xy = np.array([memory["current_internal_base_pose"][0], memory["current_internal_base_pose"][1]])
        dist_to_goal = float(np.linalg.norm(goal_xy - cur_xy))
        yaw_error = float((goal_yaw - memory["current_internal_base_pose"][2] + np.pi) % (2*np.pi) - np.pi)

        # Initialize action:
        action = Action(np.zeros_like(robot.action_space.low))
        action._arr[:len(memory["arm_joints"])] = memory["arm_joints"]

        # 1) Hard stop region
        if dist_to_goal <= COMPLETE_STOP_DISTANCE and abs(yaw_error) <= orientation_tol:
            action.set_base_motion((0.0, 0.0), "velocity")
            memory["control_phase"] = "COMPLETE"
            print(f"[STEP {memory['step_count']}] GOAL REACHED - d={dist_to_goal:.4f}, yaw={yaw_error:.4f}")
            return action

        # 2) Orientation-only (rotate in place)
        if dist_to_goal <= ORIENTATION_ONLY_DISTANCE:
            memory["control_phase"] = "ORIENTATION_ONLY"
            if abs(yaw_error) <= ORIENTATION_DEADBAND:
                #v_cmd, omega = 0.0, 0.0
                # slow nudge
                v_cmd = min(0.15, 1.5 * dist_to_goal)

                # small steering toward goal
                bearing = np.arctan2(goal_xy[1]-memory["current_internal_base_pose"][1], goal_xy[0]-memory["current_internal_base_pose"][0])
                yaw_to_goal = ((bearing - memory["current_internal_base_pose"][2] + np.pi) % (2*np.pi)) - np.pi
                omega = np.clip(0.5 * yaw_to_goal, -w_max*0.2, w_max*0.2)

                v_cmd = float(np.clip(v_cmd, 0.0, v_max))
                omega = float(np.clip(omega, -w_max, w_max))

                print(f"[STEP {memory['step_count']}] ORIENT DEADBAND - d={dist_to_goal:.3f}, yaw={yaw_error:.3f}")
            else:
                # proportional yaw control; clip with base yaw limit
                omega = float(np.clip(orientation_gain * yaw_error, -w_max, w_max))
                v_cmd = 0.0
                print(f"[STEP {memory['step_count']}] ORIENT ONLY - d={dist_to_goal:.3f}, yaw={yaw_error:.3f}, ω={omega:.3f}")

            # IK:
            omega_r = np.clip((2*v_cmd+omega*track_width)/(2*wheel_radius), -omega_max, omega_max)
            omega_l = np.clip((2*v_cmd-omega*track_width)/(2*wheel_radius), -omega_max, omega_max) 

            action.set_base_motion((omega_r, omega_l), "velocity")

            #Forward kinematics to compute internal base pose:
            v_fwd = (wheel_radius * 0.5) * (omega_r + omega_l)
            omega_fwd = (wheel_radius/track_width) * (omega_r - omega_l)
            x_next = memory["current_internal_base_pose"][0] + v_fwd * np.cos(memory["current_internal_base_pose"][2]) * dt
            y_next = memory["current_internal_base_pose"][1] + v_fwd * np.sin(memory["current_internal_base_pose"][2]) * dt
            theta_next = (memory["current_internal_base_pose"][2] + omega_fwd*dt + np.pi) % (2*np.pi) - np.pi

            #Update memory:
            memory["current_internal_base_pose"] = (x_next, y_next, theta_next)

            # action.set_base_motion((v_cmd, omega), "velocity")
            return action

        # 3) Navigation (pure pursuit-like on lookahead, with gating)
        memory["control_phase"] = "NAVIGATION"

        # advance waypoint pointer if closer to next
        path_pointer = int(memory.get("path_pointer", 0))
        waypoints = memory["path"]
        while path_pointer + 1 < len(waypoints):
            next_x, next_y = waypoints[path_pointer + 1][:2]
            current_x, current_y = waypoints[path_pointer][:2]
            if (memory["current_internal_base_pose"][0] - next_x)**2 + (memory["current_internal_base_pose"][1] - next_y)**2 <\
                            (memory["current_internal_base_pose"][0] - current_x)**2 + (memory["current_internal_base_pose"][1] - current_y)**2:
                path_pointer += 1
            else:
                break
        memory["path_pointer"] = path_pointer

        # lookahead waypoint
        lookahead_idx = min(path_pointer + max(1, int(LOOKAHEAD/0.01)), len(waypoints) - 1)
        wx, wy = waypoints[lookahead_idx][:2]

        # heading to lookahead
        alpha_W = float((np.arctan2(wy - memory["current_internal_base_pose"][1], wx - memory["current_internal_base_pose"][0]) - \
                                                                memory["current_internal_base_pose"][2] + np.pi) % (2*np.pi) - np.pi)

        # curvature -> ω, plus velocity gating by heading and distance
        la = max(0.05, min(LOOKAHEAD, dist_to_goal))
        v_cmd = vel
        v_cmd *= float(np.exp(-0.8 * abs(alpha_W)))       # slow if misaligned
        v_cmd = float(min(v_cmd, 1.5 * dist_to_goal))     # don’t overdrive when close
        v_cmd = float(np.clip(v_cmd, 0.0, v_max))

        kappa = 2.0 * np.sin(alpha_W) / la
        omega = float(np.clip(kappa * v_cmd, -w_max, w_max))

        print(f"[STEP {memory['step_count']}] NAV - d={dist_to_goal:.3f} ptr={path_pointer} look={lookahead_idx} "
              f"αW={alpha_W:.2f} v={v_cmd:.2f} ω={omega:.2f}")

        # action.set_base_motion((v_cmd, omega), "velocity")

        # IK:
        omega_r = np.clip((2*v_cmd+omega*track_width)/(2*wheel_radius), -omega_max, omega_max)
        omega_l = np.clip((2*v_cmd-omega*track_width)/(2*wheel_radius), -omega_max, omega_max) 

        action.set_base_motion((omega_r, omega_l), "velocity")

        #Forward kinematics to compute internal base pose:
        v_fwd = (wheel_radius * 0.5) * (omega_r + omega_l)
        omega_fwd = (wheel_radius/track_width) * (omega_r - omega_l)
        x_next = memory["current_internal_base_pose"][0] + v_fwd * np.cos(memory["current_internal_base_pose"][2]) * dt
        y_next = memory["current_internal_base_pose"][1] + v_fwd * np.sin(memory["current_internal_base_pose"][2]) * dt
        theta_next = (memory["current_internal_base_pose"][2] + omega_fwd*dt + np.pi) % (2*np.pi) - np.pi

        #Update memory:
        memory["current_internal_base_pose"] = (x_next, y_next, theta_next)


        return action

    def _terminal(state: State, memory: Dict, objects: Sequence[Object], params: Array) -> bool:
        # (x, y, th), _ = get_current_base_and_arm_pose(robot, state, objects, params)
        if "path" not in memory or "target_base_pose" not in memory or "path_pointer" not in memory:
            return False
        gx, gy, gth = memory["target_base_pose"]
        pos_ok = (np.hypot(gx - memory["current_internal_base_pose"][0], gy - memory["current_internal_base_pose"][1]) <= move_to_pose_tol)
        yaw_ok = (abs((gth - memory["current_internal_base_pose"][2] + np.pi) % (2*np.pi) - np.pi) <= orientation_tol)
        if pos_ok and yaw_ok:
            print(f"[TERMINAL] SUCCESS - Position error: {np.hypot(gx - memory["current_internal_base_pose"][0],\
                                                                   gy - memory["current_internal_base_pose"][1]):.4f}, "
                  f"Orientation error: {abs((gth - memory["current_internal_base_pose"][2] + np.pi) % (2*np.pi) - np.pi):.4f}")
        return pos_ok and yaw_ok

    return ParameterizedOption(
        name=name,
        types=types,
        params_space=params_space,
        policy=_policy,
        initiable=_initiable,
        terminal=_terminal,
    )